# ChartQA VLM baselines

Runs a base **vision-language** model on the ChartAudit-Fin v2.0 **test** split (180 finance and economics items, never trained on, human ChartQA answers) and scores it.

**Three baselines, run one after another by cell 9** (listed in `CONFIGS` in cell 2):
- **Zero-shot**: `("answer_only", 0)`, just the chart and the question
- **Few-shot**: `("answer_only", 2)`, two solved train examples first
- **Chain-of-thought**: `("cot", 0)`, the model writes evidence and reasoning before the answer

**How to run:** `Runtime → Run all`. Results go to `MyDrive/chartqa_vlm_baselines/` (`summary.csv`, one row per baseline, plus each run's predictions).

**Variants** (same items and targets, only the input differs):
- `image_only`: the chart image and the question. The model has to read the chart itself.
- `image_table`: the chart image plus the chart data as text (table and colours), then the question.

**Models** (all load in 4-bit):
- `Qwen/Qwen2.5-VL-3B-Instruct` (default, the approved project model)
- `Qwen/Qwen2.5-VL-7B-Instruct`
- `Qwen/Qwen2-VL-7B-Instruct`
- `meta-llama/Llama-3.2-11B-Vision-Instruct` (gated, 11B so above the ≤8B limit; comparison only, zero-shot only because it takes one image per prompt)

**Runtime:** `Runtime → Change runtime type`. An **L4 or A100** is comfortable. A free **T4** works for the 7B models in 4-bit with `BATCH_SIZE = 2`, but is slow (roughly 30–45 minutes per run on the test split).

**Secrets** (key icon in the left sidebar, turn on "Notebook access"):
- `GITHUB_TOKEN`: a GitHub token with read access to `parulrewatkar0692/chartqa-vlm-capstone-project` (private). Without it, upload the repo ZIP by hand (cell 3 explains).
- `HF_TOKEN`: a Hugging Face token (needed for Llama; accept its licence on the model page first).

**Metrics**
- **Relaxed accuracy**: numbers within 5% of gold, text and yes/no after normalising case and punctuation, unanswerable items correct when the model returns `"answer": null`.
- **JSON valid / Schema valid**: output parses as JSON / has all five fields with an allowed `answer_type`.
- **Evidence grounded**: share of the numbers the model cites as evidence that really appear in the chart's underlying data table (`input_text`). In `image_only` the model never sees that table, so this measures how well it read the chart. Only meaningful for the chain-of-thought run.

In [ ]:
# 1. Install
!pip -q install -U "transformers>=4.49" accelerate bitsandbytes pillow

In [ ]:
# 2. Settings
DATA_VERSION = "v2.0"     # ChartAudit-Fin v2.0 (blind-labelled finance + economics)
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"  # or "Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen2-VL-7B-Instruct",
                                          #    "meta-llama/Llama-3.2-11B-Vision-Instruct"
VARIANT = "image_only"   # "image_only" (chart image + question) or "image_table" (image + text table + question)
SPLIT = "test"           # "test" for the reported baseline, "val" while you experiment
CONFIGS = [("answer_only", 0), ("answer_only", 2), ("cot", 0)]   # (prompt style, shots): zero-shot, few-shot, chain-of-thought
                         # "answer_only" = just the answer; "cot" = evidence + reasoning first. Shots come from train;
                         # each shot adds one more image to the prompt (more memory). Remove a line to skip that run.
LIMIT = None             # e.g. 20 for a quick smoke test
BATCH_SIZE = 4           # 2 on a T4
MAX_NEW_TOKENS = 384     # room for evidence + reasoning in cot
LOAD_IN_4BIT = True
MAX_PIXELS = 800 * 800   # larger images are scaled down (keeps visual tokens around 800 per chart for Qwen)
SAVE_TO_DRIVE = True     # results go to MyDrive/chartqa_vlm_baselines/
SEED = 42

In [ ]:
# 3. Get the dataset (clones the private repo with GITHUB_TOKEN; falls back to a manual ZIP upload)
import os, subprocess, zipfile, glob, shutil
REPO = "parulrewatkar0692/chartqa-vlm-capstone-project"
ROOT = "/content/chartqa-vlm"
if not os.path.isdir(f"{ROOT}/data/{DATA_VERSION}"):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("No GITHUB_TOKEN secret:", e)
    if token:
        subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", ROOT], check=True)
    else:
        # Manual route: on GitHub, Code -> Download ZIP, then upload that ZIP here.
        from google.colab import files
        name, data = next(iter(files.upload().items()))
        open("/content/repo.zip", "wb").write(data)
        zipfile.ZipFile("/content/repo.zip").extractall("/content/_repo")
        shutil.move(glob.glob("/content/_repo/*")[0], ROOT)
DATA_DIR = f"{ROOT}/data/{DATA_VERSION}"
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# 4. Load the splits
import json, random, sys, collections
from PIL import Image

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def load_split(prompt_style):
    """Eval items and train items (for few-shot examples) for one prompt style."""
    sub = "answer_only/" if prompt_style == "answer_only" else ""
    items = read_jsonl(f"{DATA_DIR}/{VARIANT}/{sub}{SPLIT}.jsonl")
    train = read_jsonl(f"{DATA_DIR}/{VARIANT}/{sub}train.jsonl")
    return (items[:LIMIT] if LIMIT else items), train

eval_set, train = load_split("answer_only")
sys.path.insert(0, f"{ROOT}/scripts")
import check_teacher as CT   # evidence checker used for the teacher labels

def load_image(rel):
    img = Image.open(f"{DATA_DIR}/{rel}").convert("RGB")
    w, h = img.size
    if w * h > MAX_PIXELS:
        s = (MAX_PIXELS / (w * h)) ** 0.5
        img = img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)
    return img

print(len(eval_set), "eval items |", VARIANT, "|", collections.Counter(r["answer_type"] for r in eval_set))
load_image(eval_set[0]["image"])

In [ ]:
# 5. Prompts. The eval item's system + user turns are used as-is; few-shot examples are full
#    train conversations (image + question -> grounded JSON answer) placed before it.
# The two train examples used for the reported few-shot baseline, pinned so the result can be
# rerun exactly even if the train file changes. Delete this list to pick examples by seed.
FEW_SHOT_IDS = ["train-multi_col_20389-6093", "train-87453827004907-3340"]

def pick_shots(train, k, seed=SEED):
    by_id = {r["id"]: r for r in train}
    if 0 < k <= len(FEW_SHOT_IDS) and all(i in by_id for i in FEW_SHOT_IDS[:k]):
        return [by_id[i] for i in FEW_SHOT_IDS[:k]]
    # Otherwise: one example per answer type first (number, text, yes_no, unanswerable).
    rng = random.Random(seed)
    pool = list(train)
    rng.shuffle(pool)
    order = ["number", "text", "yes_no", "unanswerable", "number", "text"]
    shots, used = [], set()
    for t in order:
        if len(shots) == k:
            break
        for r in pool:
            if r["answer_type"] == t and r["id"] not in used:
                shots.append(r); used.add(r["id"]); break
    return shots

def strip_paths(msg):
    """Chat templates want {"type": "image"} placeholders; the PIL images are passed separately."""
    if isinstance(msg["content"], str):
        return dict(msg)
    return {"role": msg["role"], "content": [{"type": "image"} if c["type"] == "image" else c for c in msg["content"]]}

def build(item, shots):
    """Returns (messages, images) with images in the order their placeholders appear."""
    msgs, images = [strip_paths(item["messages"][0])], []
    for s in shots:
        msgs += [strip_paths(s["messages"][1]), strip_paths(s["messages"][2])]
        images.append(load_image(s["image"]))
    msgs.append(strip_paths(item["messages"][1]))
    images.append(load_image(item["image"]))
    return msgs, images

def to_prompt(processor, msgs):
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Templates without a system role: fold the system prompt into the first user turn's text.
        first = {"role": "user", "content": [dict(c) for c in msgs[1]["content"]]}
        for c in first["content"]:
            if c["type"] == "text":
                c["text"] = msgs[0]["content"] + "\n\n" + c["text"]; break
        return processor.apply_chat_template([first] + msgs[2:], tokenize=False, add_generation_prompt=True)

In [ ]:
# 6. Parse and score
import re
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None

def fields_for(prompt_style):
    return {"answer", "answer_type", "unit"} | ({"evidence", "rationale"} if prompt_style == "cot" else set())
PROMPT_STYLE, FIELDS = "answer_only", fields_for("answer_only")   # set again for each run in cell 9
TYPES = {"number", "text", "yes_no", "unanswerable"}

def relaxed_match(pred, gold, answer_type):
    if answer_type == "unanswerable":
        return pred is None or (isinstance(pred, str) and pred.strip().lower() in ("", "null", "none"))
    if pred is None:
        return False
    if answer_type == "number":
        p, g = CT.num(pred), CT.num(gold)
        if p is None or g is None:
            return False
        return abs(p - g) <= 0.05 * abs(g) if g != 0 else abs(p) < 1e-9
    return CT.norm(pred) == CT.norm(gold)

def nums(s):
    return [float(x.replace(",", "")) for x in re.findall(r"(?<![A-Za-z\d.])\d[\d,]*(?:\.\d+)?", str(s))]

def score(item, raw):
    out = extract_json(raw)
    if not isinstance(out, dict):
        out = None
    gold = json.loads(item["messages"][2]["content"])
    res = {"id": item["id"], "raw": raw, "json_valid": out is not None}
    res["schema_valid"] = bool(out) and set(out) >= FIELDS and out.get("answer_type") in TYPES \
        and (PROMPT_STYLE != "cot" or isinstance(out.get("evidence"), list))
    res["pred"] = out.get("answer") if out else None
    res["correct"] = bool(out) and relaxed_match(res["pred"], gold["answer"], item["answer_type"])
    ev = [e for e in (out or {}).get("evidence") or [] if isinstance(e, str)]
    # Evidence grounding: numbers the model cites that really are in the chart's data table
    # (the model never sees that table in image_only).
    cited = [v for e in ev for v in nums(e)]
    table = nums(item["input_text"])
    res["evidence_n"] = len(cited)
    res["evidence_ok"] = sum(any(abs(v - t) <= max(0.051, abs(t) * 0.01) for t in table) for v in cited)
    return res

In [ ]:
# 7. Load the model
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped:", e)

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=DTYPE) if LOAD_IN_4BIT else None
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, device_map="auto", quantization_config=quant,
                                                    torch_dtype=DTYPE)
model.eval()
if "llama" in MODEL_ID.lower() and any(k > 0 for _, k in CONFIGS):
    print("Llama 3.2 Vision takes one image per prompt: skipping few-shot runs.")
    CONFIGS = [(s, k) for s, k in CONFIGS if k == 0]
print(MODEL_ID, "loaded on", model.device, "| dtype", DTYPE)

In [ ]:
# 8. Helpers: generate answers (greedy, no randomness) and summarise the scores
from tqdm.auto import tqdm

@torch.no_grad()
def generate(items, shots):
    outs = []
    for i in tqdm(range(0, len(items), BATCH_SIZE)):
        built = [build(it, shots) for it in items[i:i + BATCH_SIZE]]
        prompts = [to_prompt(processor, m) for m, _ in built]
        batch = processor(text=prompts, images=[imgs for _, imgs in built], padding=True,
                          return_tensors="pt").to(model.device)
        gen = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=processor.tokenizer.pad_token_id)
        outs += processor.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

import pandas as pd

def summarise(results, items):
    df = pd.DataFrame(results).merge(
        pd.DataFrame([{k: it[k] for k in ("id", "source", "chart_type", "answer_type", "question_type")}
                      for it in items]), on="id")
    overall = {
        "n": len(df),
        "relaxed_accuracy": df.correct.mean(),
        "json_valid": df.json_valid.mean(),
        "schema_valid": df.schema_valid.mean(),
        "evidence_grounded": df.evidence_ok.sum() / max(df.evidence_n.sum(), 1),
        "refusal_accuracy": df[df.answer_type == "unanswerable"].correct.mean(),
        "answerable_accuracy": df[df.answer_type != "unanswerable"].correct.mean(),
    }
    by = {col: df.groupby(col).correct.agg(["mean", "count"]).round(3) for col in
          ("question_type", "answer_type", "source", "chart_type")}
    return df, overall, by

In [ ]:
# 9. Run all baselines in CONFIGS, print each report and save (Drive if available, else /content)
import datetime
out_dir = "/content/chartqa_vlm_baselines"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/chartqa_vlm_baselines"
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
os.makedirs(out_dir, exist_ok=True)
summary_path = f"{out_dir}/summary.csv"

summary_rows = []
for PROMPT_STYLE, k in CONFIGS:
    FIELDS = fields_for(PROMPT_STYLE)          # used by score()
    eval_set, train = load_split(PROMPT_STYLE)
    name = f"{PROMPT_STYLE}_{k}shot"
    print(f"\n===== {MODEL_ID} | {VARIANT} | {SPLIT} | {name} =====")
    raws = generate(eval_set, pick_shots(train, k))
    res = [score(it, raw) for it, raw in zip(eval_set, raws)]
    df, overall, by = summarise(res, eval_set)
    print(pd.Series(overall).round(3).to_string())
    for col, t in by.items():
        print(f"\nby {col}:\n{t.to_string()}")
    print("\nfirst raw output:", res[0]["raw"])
    # save after each run, so a disconnect does not lose finished runs
    tag = f"{MODEL_ID.split('/')[-1]}_{VARIANT}_{SPLIT}_{name}_{datetime.datetime.now():%Y%m%d-%H%M}"
    with open(f"{out_dir}/{tag}_predictions.jsonl", "w") as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    row = {"model": MODEL_ID, "variant": VARIANT, "split": SPLIT, "prompt_style": PROMPT_STYLE, "shots": k,
           **{a: round(b, 4) for a, b in overall.items()}}
    summary_rows.append(row)
    pd.DataFrame([row]).to_csv(summary_path, mode="a", header=not os.path.exists(summary_path), index=False)
print("\nsaved to", out_dir)
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
# 10. Free GPU memory before switching MODEL_ID in cell 2 and re-running cells 7-9
#     (to switch only VARIANT or CONFIGS, re-run cells 2, 4 and 9; the model stays loaded)
import gc
del model; gc.collect(); torch.cuda.empty_cache()